<a href="https://colab.research.google.com/github/Gloou-ui/NM/blob/main/%D0%9B%D0%A09_%D0%9B%D0%B8%D1%82%D0%B2%D0%B8%D0%BD_%D0%A4%D0%86%D0%A23_10_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторна робота № 9
## Рекомендаційні системи: SVD та KNNBasic

**ПІБ студента:** Литвин Мирослав  
**Група:** ФІТ3-10

**Дата:** 18.11.2027 19:20

> **Призначення:** шаблон для самостійного виконання ЛР на основі прикладу з алгоритмами `SVD` і `KNNBasic` у бібліотеці `Surprise`. Комірки з позначкою **TODO** потрібно заповнити власним кодом. Це не готовий розв’язок і не заміна офіційної умови роботи.

**Мета:** завантажити дані про оцінки фільмів, побудувати дві рекомендаційні моделі, підібрати гіперпараметри, порівняти моделі за RMSE/MAE та сформувати топ-10 рекомендацій для користувача.

**Дані:** вбудований набір `MovieLens 100K` (`ml-100k`) із бібліотеки `Surprise`.

**Послідовність роботи:** підготовка середовища → дані → розбиття → налаштування моделей → навчання → тестування → рекомендації → висновки.

## 1. Підготовка середовища

Комбінація бібліотек для **Google Colab із Python 3.13**. Встановлення без `--force-reinstall`, щоб без потреби не перевстановлювати всі залежності. Після зміни NumPy чи SciPy у вже запущеному середовищі **перезапустіть сесію** (`Runtime → Restart session`), а потім виконайте комірки нижче з початку.

In [1]:
%pip install -q \
    numpy==2.2.6 \
    scipy==1.15.3 \
    scikit-learn==1.6.1 \
    pandas==2.2.3 \
    scikit-surprise==1.1.5

In [2]:
import sys
import numpy as np
import scipy
import sklearn
import pandas as pd
import surprise

from surprise import Dataset, Reader, SVD, KNNBasic, accuracy
from surprise.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split as sklearn_train_test_split

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)
print('scikit-learn:', sklearn.__version__)
print('pandas:', pd.__version__)
print('Surprise:', surprise.__version__)

Python: 3.13.18
NumPy: 2.4.6
SciPy: 1.15.3
scikit-learn: 1.6.1
pandas: 2.2.3
Surprise: 1.1.5


## 2. Завантаження та первинний аналіз даних

1. Завантажте вбудований набір `ml-100k`.
2. Перетворіть оцінки у `pandas.DataFrame` з колонками `user_id`, `item_id`, `rating`, `timestamp`.
3. Виведіть розмір таблиці, декілька рядків, кількість унікальних користувачів та фільмів, розподіл оцінок.

**Зверніть увагу:** ідентифікатори користувачів і фільмів у `Surprise` є зовнішніми (raw) ID; вони можуть мати рядковий тип.

In [3]:
data = Dataset.load_builtin('ml-100k', prompt=False)

ratings_df = pd.DataFrame(
    data.raw_ratings,
    columns=['user_id', 'item_id', 'rating', 'timestamp']
)

print('Розмір:', ratings_df.shape)
display(ratings_df.head())

Trying to download dataset from https://files.grouplens.org/datasets/movielens/ml-100k.zip...
Done! Dataset ml-100k has been saved to /root/.surprise_data/ml-100k
Розмір: (100000, 4)


,user_id,item_id,rating,timestamp
0,196,242,3.0,881250949
1,186,302,3.0,891717742
2,22,377,1.0,878887116
3,244,51,2.0,880606923
4,166,346,1.0,886397596


In [4]:
print('Кількість унікальних користувачів:', ratings_df['user_id'].nunique())
print('Кількість унікальних фільмів:', ratings_df['item_id'].nunique())
print('Мінімальна оцінка:', ratings_df['rating'].min())
print('Максимальна оцінка:', ratings_df['rating'].max())
print('Середня оцінка:', round(ratings_df['rating'].mean(), 3))
print('Кількість пропущених значень:')
print(ratings_df.isna().sum())
print('Розподіл оцінок:')
display(ratings_df['rating'].value_counts().sort_index().rename('Кількість').to_frame())


Кількість унікальних користувачів: 943
Кількість унікальних фільмів: 1682
Мінімальна оцінка: 1.0
Максимальна оцінка: 5.0
Середня оцінка: 3.53
Кількість пропущених значень:
user_id      0
item_id      0
rating       0
timestamp    0
dtype: int64
Розподіл оцінок:


,Кількість
rating,
1.0,6110
2.0,11370
3.0,27145
4.0,34174
5.0,21201


**Спостереження:** набір `ml-100k` містить 100 000 оцінок, залишених користувачами для фільмів. Кількість унікальних користувачів і фільмів, а також середнє значення оцінки наведені в комірці вище. Оцінки є числовими та обмежені шкалою набору даних; розподіл показує, які оцінки користувачі виставляли найчастіше.


## 3. Розбиття на навчальну та тестову вибірки

Виконайте випадкове розбиття **80% / 20%** із `random_state=42` за прикладом. Для налаштування гіперпараметрів використовуйте **лише навчальну** частину. Тестова вибірка має залишатися недоторканою до фінальної оцінки.

In [5]:
train_df, test_df = sklearn_train_test_split(
    ratings_df,
    test_size=0.2,
    random_state=42
)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)
print('Навчальна вибірка:', train_df.shape)
print('Тестова вибірка:', test_df.shape)


Навчальна вибірка: (80000, 4)
Тестова вибірка: (20000, 4)


In [6]:
reader = Reader(rating_scale=data.reader.rating_scale)
train_data = Dataset.load_from_df(
    train_df[['user_id', 'item_id', 'rating']],
    reader
)
testset = list(test_df[['user_id', 'item_id', 'rating']].itertuples(index=False, name=None))
print('Кількість записів train:', len(train_df))
print('Кількість записів test:', len(testset))


Кількість записів train: 80000
Кількість записів test: 20000


## 4. Гіперпараметри моделей

Для `SVD` розгляньте кількість прихованих факторів, швидкість навчання та регуляризацію. Для `KNNBasic` — число сусідів, мінімальне число сусідів та метрику схожості.

Нижче — **початкова сітка** з аналізованого прикладу. Її можна змінити й обґрунтувати вибір.

In [7]:
param_grid_svd = {
    'n_factors': [50, 100],
    'lr_all': [0.005, 0.010],
    'reg_all': [0.02, 0.05]
}

param_grid_knn = {
    'k': [20, 40],
    'min_k': [1, 5],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [False, True]
    }
}

**Обґрунтування вибору параметрів:** для SVD параметр `n_factors` визначає кількість прихованих факторів, через які модель описує вподобання користувачів і властивості фільмів. `lr_all` задає швидкість навчання, а `reg_all` допомагає уникати перенавчання. Для KNNBasic параметр `k` визначає кількість сусідів, `min_k` — мінімальну кількість сусідів для прогнозу, `name` задає метрику схожості, а `user_based` визначає, чи порівнюються користувачі, чи фільми. Початкова сітка містить кілька помірних значень кожного параметра, щоб порівняти варіанти без надмірно великого часу обчислень.


## 5. Пошук найкращих гіперпараметрів (GridSearchCV)

Для обох моделей використайте `GridSearchCV` з метриками `rmse` та `mae` і `cv=3`. Навчайте об’єкти пошуку **лише на `train_data`**.

In [8]:
grid_search_svd = GridSearchCV(
    SVD,
    param_grid_svd,
    measures=['rmse', 'mae'],
    cv=3,
    n_jobs=-1,
    joblib_verbose=0
)
grid_search_svd.fit(train_data)
print('Пошук параметрів SVD завершено.')


Пошук параметрів SVD завершено.


In [9]:
grid_search_knn = GridSearchCV(
    KNNBasic,
    param_grid_knn,
    measures=['rmse', 'mae'],
    cv=3,
    n_jobs=-1,
    joblib_verbose=0
)
grid_search_knn.fit(train_data)
print('Пошук параметрів KNNBasic завершено.')


Пошук параметрів KNNBasic завершено.


In [10]:
for model_name, search in [('SVD', grid_search_svd), ('KNNBasic', grid_search_knn)]:
    print(f'\n{model_name}')
    print('Найкращий CV RMSE:', round(search.best_score['rmse'], 4))
    print('Параметри за RMSE:', search.best_params['rmse'])
    print('Найкращий CV MAE:', round(search.best_score['mae'], 4))
    print('Параметри за MAE:', search.best_params['mae'])

cv_comparison = pd.DataFrame([
    {'Model': 'SVD', 'CV RMSE': grid_search_svd.best_score['rmse'],
     'CV MAE': grid_search_svd.best_score['mae'],
     'Best parameters (RMSE)': str(grid_search_svd.best_params['rmse'])},
    {'Model': 'KNNBasic', 'CV RMSE': grid_search_knn.best_score['rmse'],
     'CV MAE': grid_search_knn.best_score['mae'],
     'Best parameters (RMSE)': str(grid_search_knn.best_params['rmse'])}
]).sort_values('CV RMSE')
display(cv_comparison)



SVD
Найкращий CV RMSE: 0.944
Параметри за RMSE: {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}
Найкращий CV MAE: 0.7449
Параметри за MAE: {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}

KNNBasic
Найкращий CV RMSE: 1.0278
Параметри за RMSE: {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}
Найкращий CV MAE: 0.8154
Параметри за MAE: {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}


,Model,CV RMSE,CV MAE,Best parameters (RMSE)
0,SVD,0.943994,0.744946,"{'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0..."
1,KNNBasic,1.027815,0.815422,"{'k': 40, 'min_k': 1, 'sim_options': {'name': ..."


**Аналіз результатів кросвалідації:** порівняйте рядки таблиці, сформованої попередньою коміркою. Менше значення RMSE означає меншу середню квадратичну похибку прогнозів, а менше MAE — меншу середню абсолютну похибку. Для подальших рекомендацій обирається модель із найменшим CV RMSE; тестові дані під час цього вибору не використовуються.


## 6. Навчання остаточних моделей

Побудуйте `trainset = train_data.build_full_trainset()`. Створіть нові екземпляри SVD і KNNBasic з параметрами, обраними за **CV RMSE**, після чого навчіть їх на **всій навчальній вибірці**.

In [11]:
best_svd_params = grid_search_svd.best_params['rmse']
best_knn_params = grid_search_knn.best_params['rmse']

trainset = train_data.build_full_trainset()
best_svd = SVD(**best_svd_params, random_state=42)
best_knn = KNNBasic(**best_knn_params)

best_svd.fit(trainset)
best_knn.fit(trainset)
print('Обидві моделі навчено на повній навчальній вибірці.')


Computing the cosine similarity matrix...
Done computing similarity matrix.
Обидві моделі навчено на повній навчальній вибірці.


## 7. Оцінювання на незалежній тестовій вибірці

Отримайте передбачення для `testset` методами `.test()` обох моделей. Обчисліть `RMSE` та `MAE` за допомогою `surprise.accuracy`.

**Методична примітка:** вибирайте модель для рекомендацій за результатами **кросвалідації**, а незалежну тестову вибірку використовуйте для фінальної оцінки якості, а не для повторного підбору.

In [12]:
svd_predictions = best_svd.test(testset)
knn_predictions = best_knn.test(testset)

svd_rmse = accuracy.rmse(svd_predictions, verbose=False)
svd_mae = accuracy.mae(svd_predictions, verbose=False)
knn_rmse = accuracy.rmse(knn_predictions, verbose=False)
knn_mae = accuracy.mae(knn_predictions, verbose=False)

print(f'SVD — RMSE: {svd_rmse:.4f}, MAE: {svd_mae:.4f}')
print(f'KNNBasic — RMSE: {knn_rmse:.4f}, MAE: {knn_mae:.4f}')


SVD — RMSE: 0.9192, MAE: 0.7234
KNNBasic — RMSE: 1.0120, MAE: 0.7984


In [13]:
results = pd.DataFrame([
    {'Model': 'SVD', 'RMSE': svd_rmse, 'MAE': svd_mae},
    {'Model': 'KNNBasic', 'RMSE': knn_rmse, 'MAE': knn_mae}
]).sort_values('RMSE').reset_index(drop=True)
display(results)


,Model,RMSE,MAE
0,SVD,0.919175,0.723355
1,KNNBasic,1.011953,0.798421


**Порівняння з CV:** зіставте тестові RMSE та MAE з найкращими значеннями кросвалідації у таблиці вище. Невелика різниця між CV і тестом свідчить, що якість моделі на відкладених даних близька до результатів перехресної перевірки. Помітна різниця може вказувати на особливості випадкового розбиття або нестабільність моделі.


## 8. Топ-10 рекомендацій для користувача

1. Оберіть модель із кращим **CV RMSE**.
2. Задайте `user_id` (вихідний приклад використовує `'196'`).
3. Отримайте фільми, які цей користувач уже оцінив у навчальній вибірці.
4. Для решти фільмів обчисліть прогнозований рейтинг `.predict(uid=..., iid=...).est`.
5. Виведіть **10 фільмів із найбільшим прогнозованим рейтингом**.

**Важливо:** не пропонуйте користувачу фільми, які він уже оцінив у навчальних даних. Для коректного перетворення внутрішніх та зовнішніх ID використовуйте `trainset.to_inner_uid()`, `trainset.to_raw_iid()`.

In [15]:
user_id = '196'

if grid_search_svd.best_score['rmse'] <= grid_search_knn.best_score['rmse']:
    recommendation_model = best_svd
    recommendation_model_name = 'SVD'
else:
    recommendation_model = best_knn
    recommendation_model_name = 'KNNBasic'

print('Обрана за CV RMSE модель:', recommendation_model_name)

try:
    inner_uid = trainset.to_inner_uid(user_id)
except ValueError:
    inner_uid = None

if inner_uid is None:
    print(f'Користувача з ID {user_id} немає у навчальній вибірці. Змініть user_id на наявний ID.')
else:
    # Список фільмів, які користувач уже оцінив у train
    rated_inner_iids = {inner_iid for inner_iid, _ in trainset.ur[inner_uid]}
    all_inner_iids = set(range(trainset.n_items))
    unseen_inner_iids = all_inner_iids - rated_inner_iids

    recommendations = []
    for inner_iid in unseen_inner_iids:
        raw_iid = trainset.to_raw_iid(inner_iid)
        predicted_rating = recommendation_model.predict(user_id, raw_iid).est
        recommendations.append((raw_iid, predicted_rating))

    top10_recommendations = pd.DataFrame(
        recommendations,
        columns=['item_id', 'predicted_rating']
    ).sort_values('predicted_rating', ascending=False).head(10).reset_index(drop=True)

    print(f'Топ-10 рекомендацій для користувача {user_id}:')
    display(top10_recommendations)


Обрана за CV RMSE модель: SVD
Топ-10 рекомендацій для користувача 196:


,item_id,predicted_rating
0,320,4.490358
1,64,4.470680
2,480,4.413710
3,313,4.401920
4,513,4.399624
5,483,4.398092
6,318,4.392102
7,656,4.385379
8,22,4.377388
9,272,4.376010


## 9. Перевірка виконання

Позначте після реалізації:

- [ ] Дані `ml-100k` завантажено та перевірено.
- [ ] Є розбиття 80/20 з фіксованим `random_state`.
- [ ] `Reader`, `train_data` і `testset` створені коректно.
- [ ] GridSearchCV для **обох** моделей виконано лише на `train_data`.
- [ ] Для SVD і KNNBasic показано параметри та результати CV.
- [ ] Моделі навчено на повній навчальній частині.
- [ ] На незалежній тестовій вибірці розраховано **RMSE та MAE** для обох моделей.
- [ ] Наведено порівняльну таблицю та інтерпретацію.
- [ ] Сформовано топ-10 рекомендацій без уже оцінених фільмів.
- [ ] Усі комірки виконуються послідовно після перезапуску середовища.

## 10. Висновки

Заповніть **власними результатами**:

1. Яку модель обрано за кросвалідацією і чому?
2. Які найкращі гіперпараметри отримано для обох моделей?
3. Які підсумкові значення RMSE та MAE на тестовій вибірці?
4. Чи є різниця між результатами CV та тестування?
5. Які обмеження мають отримані рекомендації (наприклад, нові користувачі/фільми)?

**Висновок:** _Напишіть 5–8 речень з опорою на отримані числові результати._

---
**Примітка до шаблону:** вихідний студентський ноутбук містив блок «Доцільна реалізація» з SVD, KNNBasic і топ-10 рекомендаціями. Тому тут відтворено ці етапи; додаткових офіційних вимог, не наведених у зразку, не припускаємо.

**Додаткове завдання. Написати власний рекомендаційний алгоритм з нуля**

In [16]:
best_cv_name = 'SVD' if grid_search_svd.best_score['rmse'] <= grid_search_knn.best_score['rmse'] else 'KNNBasic'
best_cv_rmse = min(grid_search_svd.best_score['rmse'], grid_search_knn.best_score['rmse'])
best_cv_mae = grid_search_svd.best_score['mae'] if best_cv_name == 'SVD' else grid_search_knn.best_score['mae']

print(f"У лабораторній роботі було створено рекомендаційні моделі SVD та KNNBasic на основі набору MovieLens 100K, який завантажується онлайн через бібліотеку Surprise. Дані було розділено на навчальну (80%) і тестову (20%) вибірки з random_state=42. Для обох моделей виконано підбір гіперпараметрів методом GridSearchCV із трьома блоками кросвалідації. За найменшим CV RMSE обрано модель {best_cv_name} із RMSE={best_cv_rmse:.4f} та MAE={best_cv_mae:.4f}. На незалежній тестовій вибірці SVD отримала RMSE={svd_rmse:.4f} і MAE={svd_mae:.4f}, а KNNBasic — RMSE={knn_rmse:.4f} і MAE={knn_mae:.4f}. Для користувача {user_id} сформовано персональний список фільмів, які він ще не оцінював у навчальних даних. Отримані рекомендації залежать від наявної історії оцінок, тому для нових користувачів або фільмів без історії взаємодій якість може бути обмеженою.")


У лабораторній роботі було створено рекомендаційні моделі SVD та KNNBasic на основі набору MovieLens 100K, який завантажується онлайн через бібліотеку Surprise. Дані було розділено на навчальну (80%) і тестову (20%) вибірки з random_state=42. Для обох моделей виконано підбір гіперпараметрів методом GridSearchCV із трьома блоками кросвалідації. За найменшим CV RMSE обрано модель SVD із RMSE=0.9440 та MAE=0.7449. На незалежній тестовій вибірці SVD отримала RMSE=0.9192 і MAE=0.7234, а KNNBasic — RMSE=1.0120 і MAE=0.7984. Для користувача 196 сформовано персональний список фільмів, які він ще не оцінював у навчальних даних. Отримані рекомендації залежать від наявної історії оцінок, тому для нових користувачів або фільмів без історії взаємодій якість може бути обмеженою.
